# Correlation

**Notebook 5 of 6.** `04_descriptive_statistics.ipynb` described columns one at a time.
This one takes them two at a time and asks how strongly they are related.

One idea runs through all of it. A relationship between two columns can be measured
whatever their types, and every measure here reports its strength on a scale topping out
at `1`. They are not interchangeable, though: the measures for two numerical columns run
from `-1` to `1` and carry a direction, while the two for categorical pairs run from `0`
to `1` and have no direction to carry. Which measure you use, and which plot you draw,
both follow from the types of the two columns, and the next section sets out the whole
map.

First the libraries this notebook needs.

In [ ]:
# data analysis
import numpy as np
import pandas as pd

# visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# statistics
from scipy import stats

sns.set_style("whitegrid")

Then the data, the same cleaned file `04_descriptive_statistics.ipynb` worked on. The gaps
are still in it, and every correlation below drops rows pairwise as it goes.

In [ ]:
# the cleaned data from 01_data_cleaning
df_weather = pd.read_csv("../data/seattle-weather_clean.csv", parse_dates=["date"])
df_weather.head()

In [ ]:
# the numerical columns, grouped as in 03_descriptive_statistics
numerical_features = ["precipitation", "temp_max", "temp_min", "wind"]
categorical_features = ["weather"]

## Choosing the plot and the measure

Picking a correlation measure and picking a plot are **the same decision**, made from the
types of the two columns. This notebook works through the three cases in order, and this
table is the summary you should keep:

| Column A | Column B | Plot | Measure |
|---|---|---|---|
| numerical | - | histogram, boxplot | mean, median, std, IQR |
| categorical | - | **bar plot** | counts, proportions, mode |
| numerical | numerical | **scatter plot**, pairplot | Pearson `r`, Spearman `rho`, Kendall `tau` |
| categorical | categorical | **grouped bar plot**, heatmap of the crosstab | Cramer's V |
| categorical | numerical | **boxplot**, violin plot, KDE by category | correlation ratio `eta` |

The first two rows are the one-variable case from `04_descriptive_statistics.ipynb`. The
other three are this notebook.

**Always draw the plot.** A coefficient compresses a relationship into one number, and the
compression is lossy in ways that matter: it cannot show you a curve, a cluster, or the
three points that are inventing the whole result. The optional `06_mutual_information.ipynb` is
entirely about a relationship that no coefficient in this table can see, and one line plot
makes it obvious.

## 1. Two numerical columns: Pearson and Spearman

Now to relationships. For two numerical columns the standard measure is the
**Pearson correlation coefficient**, usually written `r`:

$$
r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum (x_i - \bar{x})^2 \sum (y_i - \bar{y})^2}}
$$

It runs from `-1` to `+1`:

- `+1` means the points lie exactly on a rising straight line.
- `0` means no *linear* relationship.
- `-1` means they lie exactly on a falling straight line.

The word **linear** is doing a great deal of work in that sentence, and the optional
`06_mutual_information.ipynb` is about what it hides.

The **Spearman rank correlation**, `rho`, is Pearson applied to the *ranks* instead of the
values. It measures whether the relationship is **monotonic**, that is, whether y
consistently rises as x rises, without requiring a straight line. It also ignores how
extreme the extreme values are, which makes it far more robust on skewed columns like
`precipitation`.

**Kendall's tau** is the third of the standard trio. It also works on ranks, but instead of
correlating them it counts pairs of rows: a pair is *concordant* if both columns move the
same way between them, *discordant* if they move opposite ways. Tau is the difference
between those counts, scaled to `-1` to `1`. It is more robust than Spearman when the
dataset is small or has many tied values, and it is almost always the smallest of the
three numbers, so never read a low tau as a weak relationship without comparing like with
like.

All three are one argument apart in pandas: `.corr(method="pearson")`, `"spearman"` or
`"kendall"`.

Before computing anything, look at all the pairs at once. `sns.pairplot` draws a scatter
plot for every pair of numerical columns and a distribution down the diagonal. It is the
most conventional first move in an EDA, and it often shows in one picture what a table of
coefficients can only hint at.

In [ ]:
# every pair of numerical columns at once, distributions down the diagonal
sns.pairplot(df_weather[numerical_features], plot_kws={"s": 8, "alpha": 0.4}, height=1.6);

Now the numbers behind those pictures.

In [ ]:
# Pearson correlation between every pair of numerical columns
correlations = df_weather[numerical_features].corr()
correlations.round(3)

A heatmap makes the same table readable at a glance. Two conventions are worth copying:

- **Fix the colour scale** with `vmin=-1` and `vmax=1`. Left to itself the scale stretches
  to whatever is in the data, and a table of weak correlations comes out looking dramatic.
- **Mask the upper triangle.** A correlation matrix is symmetric, so half of it is the same
  information twice, and a diagonal of `1.0` is not information at all.

In [ ]:
# the same table as a heatmap, upper triangle and diagonal hidden
mask = np.triu(np.ones_like(correlations, dtype=bool))

plt.figure(figsize=(4.5, 3.5), dpi=100)
sns.heatmap(
    data=correlations,
    mask=mask,
    cmap="coolwarm",
    linecolor="white",
    linewidth=1,
    annot=True,
    fmt=".2f",
    vmin=-1,
    vmax=1,
)
plt.title("Pearson correlation");

The strongest pair by far is `temp_max` against `temp_min`, which should be reassuring
rather than surprising: warm days have warm nights. Always look at the scatter plot behind
a correlation, because a single number cannot show you shape.

In [ ]:
# the relationship behind the strongest correlation
plt.figure(figsize=(4.5, 4), dpi=100)
sns.scatterplot(x=df_weather.temp_min, y=df_weather.temp_max, s=12, alpha=0.5)
plt.title("temp_min against temp_max");

> **Exercise:**
>
> Compute all three coefficients, Pearson, Spearman and Kendall, between `precipitation`
> and `wind`. Print them side by side, then draw the scatter plot.
>
> The first two will come out close to each other, and Kendall noticeably lower. In a
> comment, say what the first of those facts tells you about the relationship, what it
> would have meant if they had come out far apart, and why the third being lower is not a
> disagreement.

<details><summary>
Click here for a hint...
</summary>

`df.corr(method="spearman")` takes a method argument, and so does the Series version
`df["a"].corr(df["b"], method="spearman")`. Missing values are dropped pairwise, so you do
not need to handle them yourself.
</details>

In [ ]:
# all three coefficients for precipitation against wind
for method in ["pearson", "spearman", "kendall"]:
    print(f"{method:9}: {df_weather.precipitation.corr(df_weather.wind, method=method):+.3f}")

# the scatter plot
plt.figure(figsize=(4.5, 4), dpi=100)
sns.scatterplot(x=df_weather.wind, y=df_weather.precipitation, s=12, alpha=0.5)
plt.title("wind against precipitation");

# Pearson and Spearman agree closely (0.317 against 0.327), so the relationship is not only
# monotonic but close to linear, and it is not being driven by the extreme wet days.
# Had Spearman come out much higher than Pearson, that would suggest a consistent but
# curved relationship, or that a few extreme points were dragging Pearson down. Had
# Pearson come out much higher, a small number of extreme points would likely be
# inventing the correlation on their own.
#
# Kendall comes out lower, about 0.24, and that is not a disagreement. Tau is built from
# counts of concordant and discordant pairs, so it sits on a different scale from the
# other two and is routinely smaller for the same relationship. Compare tau against tau.

## 2. Two categorical columns: Cramer's V

Pearson needs numbers. It cannot tell you whether `weather` is related to the time of year,
because neither of those is a number.

For two categorical columns the starting point is a **contingency table**, also called a
crosstab: a grid counting how often each pair of categories occurs together.

This dataset has only one categorical column, so first derive a second one. You parsed
`date` into a real datetime in `02_data_cleaning.ipynb`, which means the calendar is now available through
the `.dt` accessor.

In [ ]:
# derive month and season from the date column parsed in 01_data_cleaning
df_weather["month"] = df_weather.date.dt.month
df_weather["season"] = df_weather.month.map({
    12: "winter", 1: "winter", 2: "winter",
    3: "spring", 4: "spring", 5: "spring",
    6: "summer", 7: "summer", 8: "summer",
    9: "autumn", 10: "autumn", 11: "autumn",
})
df_weather[["date", "month", "season", "weather"]].head()

> **Note.** Deriving a new column from an existing one is **feature engineering**, and it is often
> the most valuable thing an EDA produces. `season` is not in the raw file. It is a claim
> you are making about what matters, and the next few cells test whether the claim is any
> good.

In [ ]:
# how often does each weather type occur in each season
crosstab = pd.crosstab(df_weather.season, df_weather.weather)
crosstab

In [ ]:
# the same counts as a plot
sns.countplot(data=df_weather, x="season", hue="weather")
plt.title("weather type by season");

A grouped bar plot shows the counts, but the groups have very different sizes, so it is
hard to tell a real association from the simple fact that summer has more sunny days
because Seattle has more sunny days overall.

**Normalising the crosstab by row fixes that.** Each row then sums to 1, so you are
comparing the *composition* of each season rather than its size.

In [ ]:
# what share of each season's days falls into each weather type
shares = pd.crosstab(df_weather.season, df_weather.weather, normalize="index")

plt.figure(figsize=(6, 3), dpi=100)
sns.heatmap(shares, cmap="YlOrBr", annot=True, fmt=".2f", linewidth=1,
            linecolor="white", vmin=0, vmax=1)
plt.title("share of each season's days, by weather type");

Now the association is visible: summer is `0.73` sun against winter's `0.29`, while fog
runs the other way, `0.12` in summer against `0.35` in winter. That is what Cramer's V is
about to put a number on.

The table shows a relationship, but "shows a relationship" is not a number you can compare
against another pair of columns.

The conventional starting point is the **chi-squared test of independence**. It compares
the counts you observed against the counts you would expect if the two columns were
unrelated, and returns a statistic that grows as those two diverge. Its weakness is that
the statistic also grows with the size of the table and the number of rows, so it cannot
be compared across datasets.

**Cramer's V** fixes that, scaling the same statistic into a single value between `0` and
`1`:

$$
V = \sqrt{\frac{\chi^2 / n}{\min(k - 1, r - 1)}}
$$

where $\chi^2$ is the chi-squared statistic of the table, $n$ is the number of
observations, and $r$ and $k$ are its numbers of rows and columns. `0` means no
association at all, `1` means knowing one column tells you the other exactly.

The version below adds the **Bergsma-Wicher correction**, which removes a bias that makes
plain Cramer's V look too large when the table has many cells relative to the number of
rows.

In [ ]:
def cramers_v(column_a, column_b):
    """Cramer's V between two categorical Series, with the Bergsma-Wicher correction.

    Args:
        column_a: a categorical pandas Series.
        column_b: a categorical pandas Series of the same length.

    Returns:
        A float in [0, 1]. 0 is no association, 1 is a perfect one.
    """
    crosstab = pd.crosstab(column_a, column_b)
    chi2 = stats.chi2_contingency(crosstab).statistic
    n = crosstab.to_numpy().sum()
    r, k = crosstab.shape

    # the corrections below shrink chi2 and the table size towards what you would
    # expect by chance alone, so small tables are not flattered
    phi2_corrected = max(0, chi2 / n - (k - 1) * (r - 1) / (n - 1))
    r_corrected = r - (r - 1) ** 2 / (n - 1)
    k_corrected = k - (k - 1) ** 2 / (n - 1)

    return np.sqrt(phi2_corrected / min(k_corrected - 1, r_corrected - 1))

> **Exercise:**
>
> Use `cramers_v` to measure the association between `season` and `weather`. Then use it on
> `weather` against itself as a sanity check, and say in a comment what value you expect
> and why.

<details><summary>
Click here for a hint...
</summary>

A column is perfectly associated with itself, so the sanity check should return the top of
the scale. If it does not, the function is wrong.
</details>

In [ ]:
# Cramer's V between season and weather
print(f"season  vs weather : {cramers_v(df_weather.season, df_weather.weather):.3f}")

# the sanity check: a column against itself must be a perfect association, so 1.0
print(f"weather vs weather : {cramers_v(df_weather.weather, df_weather.weather):.3f}")

About `0.21`. A real association, but a modest one: knowing the season narrows down the
weather without deciding it, which matches the countplot. Seattle is rainy most of the
year.

> **Warning.** You will see people convert categories to numbers, `sun` to `0`, `rain` to `1`, `fog` to
> `2`, and then run Pearson on the result. **Do not do this.** It invents an order and a
> spacing the categories never had, so the answer changes if you happen to relabel them in
> a different order. Pearson on a label-encoded column is not a weak measurement, it is a
> meaningless one.

## 3. One categorical and one numerical column: the correlation ratio

The remaining case is a mixed pair, such as `season` against `temp_max`. Pearson cannot
take the categorical column, and Cramer's V cannot take the numerical one.

The **correlation ratio**, written `eta`, asks a simple question: **how much of the
variation in the numbers is explained by which group a row belongs to?**

$$
\eta = \sqrt{\frac{\sum_g n_g (\bar{y}_g - \bar{y})^2}{\sum_i (y_i - \bar{y})^2}}
$$

The numerator is the variation *between* group means. The denominator is the total
variation. So `eta` is `0` when every group has the same mean, and approaches `1` when the
values inside each group are all but identical and the groups differ.

Like Cramer's V, `eta` runs from `0` to `1` and is unsigned: there is no direction to
report, because the categories have no order to be higher or lower along. That is the
one thing to carry over from the Pearson section, where `r` ran from `-1` to `1` and
the sign was half the answer.

In [ ]:
def correlation_ratio(categories, values):
    """The correlation ratio eta between a categorical and a numerical Series.

    Args:
        categories: a categorical pandas Series.
        values: a numerical pandas Series of the same length.

    Returns:
        A float in [0, 1]. 0 means group membership explains nothing about the values.
    """
    paired = pd.DataFrame({"group": categories, "value": values}).dropna()
    grand_mean = paired.value.mean()
    grouped = paired.groupby("group").value

    # variation between the group means, weighted by how big each group is
    between = (grouped.count() * (grouped.mean() - grand_mean) ** 2).sum()
    # total variation, ignoring the groups entirely
    total = ((paired.value - grand_mean) ** 2).sum()

    return np.sqrt(between / total)

The plot that goes with this measure is the **boxplot**, one box per category. A high `eta`
looks like boxes sitting at clearly different heights; a low one looks like boxes at much
the same height. A **violin plot** or a **KDE per category** shows the same thing with the
shape of each group's distribution included, which matters when a group is skewed or has
two peaks.

Here are both, for a pair with a strong relationship and a pair with a weak one.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

# strong: precipitation really does differ by weather type
sns.boxplot(data=df_weather, x="weather", y="precipitation",
            order=["sun", "fog", "drizzle", "rain", "snow"], ax=axes[0])
axes[0].set_title("precipitation by weather  (eta = 0.44)")

# weak: wind barely differs
sns.boxplot(data=df_weather, x="weather", y="wind",
            order=["sun", "fog", "drizzle", "rain", "snow"], ax=axes[1])
axes[1].set_title("wind by weather")

plt.tight_layout();

In [ ]:
# the same comparison as distributions rather than summaries
plt.figure(figsize=(6, 3.5), dpi=100)
sns.kdeplot(data=df_weather, x="temp_max", hue="season",
            hue_order=["winter", "spring", "summer", "autumn"], fill=True, alpha=0.3)
plt.title("temp_max by season, one curve per category");

The KDE makes the point the correlation ratio is measuring: four curves sitting in
different places means season explains a lot of the variation in `temp_max`. Had they all
sat on top of each other, `eta` would have been near zero.

In [ ]:
# how much of the variation in precipitation is explained by the weather label
print(f"weather vs precipitation : {correlation_ratio(df_weather.weather, df_weather.precipitation):.3f}")

> **Exercise:**
>
> Compute the correlation ratio between `season` and `temp_max`, and between `weather` and
> `wind`. Then draw a boxplot of `temp_max` grouped by `season` and check that the number
> and the picture agree.

<details><summary>
Click here for a hint...
</summary>

`sns.boxplot(data=df_weather, x="season", y="temp_max")` draws the plot. A high `eta`
should show as boxes sitting at clearly different heights, a low one as boxes at much the
same height.
</details>

In [ ]:
# correlation ratio for the two pairs
print(f"season  vs temp_max : {correlation_ratio(df_weather.season, df_weather.temp_max):.3f}")
print(f"weather vs wind     : {correlation_ratio(df_weather.weather, df_weather.wind):.3f}")

# the boxplot
plt.figure(figsize=(6, 3.5), dpi=100)
sns.boxplot(data=df_weather, x="season", y="temp_max",
            order=["winter", "spring", "summer", "autumn"])
plt.title("temp_max by season");

`season` against `temp_max` comes out near `0.78`, much the strongest relationship you have
measured, and the boxplot shows why: the four boxes barely overlap. `weather` against
`wind` is near `0.24`, and boxes drawn for that pair would sit at almost the same height.

> **Note.** Squaring the correlation ratio gives `eta^2`, the **proportion of variance explained** by
> group membership. It is the effect size behind **ANOVA**, the hypothesis test that compares
> the means of several groups. `eta` tells you how large the difference between groups is;
> ANOVA tells you whether a difference that size could plausibly have arisen by chance.

## Saving your work

`month` and `season` were not in the raw file. You derived them, and the optional fifth
notebook needs them, so write the enriched data out. This is also the shape your project
data should be in at the end of an EDA: cleaned, plus whatever you worked out it needed.

In [ ]:
# hand the derived month and season columns on to 05_mutual_information
df_weather.to_csv("../data/seattle-weather_features.csv", index=False)

print(f"saved {df_weather.shape[0]} rows and {df_weather.shape[1]} columns")

## Wrapping up

You have cleaned a raw file, decided what to do about its gaps, described its columns one
at a time and then two at a time. Together those are an exploratory data analysis, and the
method transfers directly to any other dataset.

**The plot and the measure, by column type:**

| Column A | Column B | Plot | Measure | Range | Assumes |
|---|---|---|---|---|---|
| numerical | - | histogram, boxplot | mean, median, std, IQR | - | - |
| categorical | - | bar plot | counts, proportions, mode | - | - |
| numerical | numerical | scatter, pairplot | Pearson `r` | -1 to 1 | a straight line |
| numerical | numerical | scatter, pairplot | Spearman `rho`, Kendall `tau` | -1 to 1 | a consistent direction |
| categorical | categorical | grouped bar, crosstab heatmap | Cramer's V | 0 to 1 | nothing |
| categorical | numerical | boxplot, violin, KDE | correlation ratio `eta` | 0 to 1 | nothing |

**And the habits worth keeping:**

- Name the type of every variable before computing anything.
- When the mean and the median disagree, trust neither until you have seen the histogram.
- An outlier rule produces candidates, not verdicts.
- Always draw the plot behind a coefficient. One number cannot show you a curve, a cluster,
  or the three points inventing the whole result.
- A correlation of zero means no relationship **of that shape**, not no relationship.
- Say what you did to the missing values, because it changed your answers.

That last pair of habits is where the optional `06_mutual_information.ipynb` picks up. It
takes one relationship in this dataset that every measure in the table above scores at
roughly zero, and one that your own imputation choices visibly bent, and measures both.